# Maximizing Revenue for Taxi Cab Drivers through Payment Type Analysis


## Problem Statement
### In the fast-paced taxi booking sector, making the most of revenue is essential for long-term success and driver happiness. Our goal is to use data-driven insights to maximise revenue streams for taxi drivers in order to meet this need. Our research aims to determine whether payment methods have an impact on fare pricing by focusing on the relationship between payment type and fare amount.

## Objective
### This project's main goal is to run an A/B test to examine the relationship between the total fare and the method of payment. We use Python hypothesis testing and descriptive statistics to extract useful information that can help taxi drivers generate more cash. In particular, we want to find out if there is a big difference in the fares for those who pay with credit cards versus those who pay with cash.

## Research Question
### Is there a relationship between total fare amount and payment type and can we nudge customers towards payment methods that generate higher revenue for drivers, without negatively impacting customer experience?

## Insatlling Libraries

In [1]:
! pip install pandas matplotlib seaborn scipy statsmodels

Defaulting to user installation because normal site-packages is not writeable


## Importing Libraries

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

print("All libraries imported successfully!")

All libraries imported successfully!


## Loading Data Set

In [3]:
taxi_data = pd.read_csv("taxi_data_2020.csv")

In [4]:
taxi_data.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
0,1.0,2020 Jan 01 12:28:15 AM,2020 Jan 01 12:33:03 AM,1.0,1.2,1.0,N,238,239,1.0,6.0,3.0,0.5,1.47,0.0,0.3,11.27,2.5
1,1.0,2020 Jan 01 12:35:39 AM,2020 Jan 01 12:43:04 AM,1.0,1.2,1.0,N,239,238,1.0,7.0,3.0,0.5,1.50,0.0,0.3,12.3,2.5
2,1.0,2020 Jan 01 12:47:41 AM,2020 Jan 01 12:53:52 AM,1.0,0.6,1.0,N,238,238,1.0,6.0,3.0,0.5,1.00,0.0,0.3,10.8,2.5
3,1.0,2020 Jan 01 12:55:23 AM,2020 Jan 01 01:00:14 AM,1.0,0.8,1.0,N,238,151,1.0,5.5,0.5,0.5,1.36,0.0,0.3,8.16,0.0
4,2.0,2020 Jan 01 12:01:58 AM,2020 Jan 01 12:04:16 AM,1.0,0.0,1.0,N,193,193,2.0,3.5,0.5,0.5,0.00,0.0,0.3,4.8,0.0


In [5]:
taxi_data.tail(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge
632170,2.0,2020 Jan 04 06:20:42 PM,2020 Jan 04 06:28:08 PM,1.0,1.31,1.0,N,90,249,1.0,6.5,0.0,0.5,1.96,0.0,0.3,11.76,2.5
632171,2.0,2020 Jan 04 06:30:07 PM,2020 Jan 04 06:37:55 PM,1.0,1.45,1.0,N,249,186,2.0,7.5,0.0,0.5,0.00,0.0,0.3,10.8,2.5
632172,2.0,2020 Jan 04 06:54:23 PM,2020 Jan 04 07:15:42 PM,1.0,4.44,1.0,N,125,40,2.0,18.0,0.0,0.5,0.00,0.0,0.3,21.3,2.5
632173,1.0,2020 Jan 04 06:32:54 PM,2020 Jan 04 06:54:44 PM,1.0,6.50,1.0,N,132,258,2.0,21.0,0.0,0.5,0.00,0.0,0.3,21.8,0.0
632174,1.0,2020 Jan 04 06:07:38 PM,2020 Jan 04 06:21:50 PM,1.0,2.60,1.0,N,107,50,1.0,11.5,2.5,0.5,2.95,0.0,0.3,17.75,2.5


## Exploratory Data Analysis

In [6]:
# rows and columns of the dataset

taxi_data.shape

(632175, 18)

In [7]:
# checking datatypes of all columns
taxi_data.dtypes

VendorID                 float64
tpep_pickup_datetime      object
tpep_dropoff_datetime     object
passenger_count          float64
trip_distance            float64
RatecodeID               float64
store_and_fwd_flag        object
PULocationID               int64
DOLocationID               int64
payment_type             float64
fare_amount               object
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amount             float64
improvement_surcharge    float64
total_amount              object
congestion_surcharge     float64
dtype: object

In [ ]:
# converting pickup and drop column datatype into datetime
taxi_data['tpep_pickup_datetime'] = pd.to_datetime(taxi_data['tpep_pickup_datetime'])
taxi_data['tpep_dropoff_datetime'] = pd.to_datetime(taxi_data['tpep_dropoff_datetime'])

In [ ]:
# checking datatypes after updating datatypes
taxi_data.dtypes

In [ ]:
# we need to add some columns (features) as needed
# Calculaating duration from pichup and dropoff datetime in minutes

# subtracting the pickup time from dropoff time to get duration
taxi_data["duration"] = taxi_data['tpep_dropoff_datetime']- taxi_data['tpep_pickup_datetime']

# converting into minutes
taxi_data["duration"] = taxi_data['duration'].dt.total_seconds()/60

In [ ]:
taxi_data

### There are so many columns in the dataset, but as per our problem statement, we only require some fields from the original data. Rest columns are nothing but the unwanted columns for this study. So we will simply remove those columns. 

### As the problem statement is revolving around the payment type, fare amount, and any other factor influencing the fare amount, we will filter the data to have only passenger count, trip distance, payment type, fare amount and duration of the trip.

In [ ]:
# Removing unwanted columns

taxi_data.drop(['VendorID','tpep_pickup_datetime','tpep_dropoff_datetime',
                'RatecodeID','store_and_fwd_flag','PULocationID','DOLocationID','extra',
                'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge'
                ,'total_amount','congestion_surcharge'],
              axis = 1, inplace = True)

In [ ]:
# filtered data with relevant column essential for analysis
taxi_data.head()

In [ ]:
taxi_data

In [ ]:
# check for missing values
taxi_data.isnull().sum()

In [ ]:
# percentage of misssing data to the total record of the data
print(f"Missing data {round(1176/taxi_data.shape[0]*100,2)} %")

In [ ]:
# as the data has only 0.19% of data missing, we can simply drop the records with missing values.
taxi_data.dropna(inplace = True)

In [ ]:
taxi_data

In [ ]:
# changing the passenger count and payment type data type to integer as they are present in float
taxi_data['passenger_count'] = taxi_data['passenger_count'].astype('int64')
taxi_data['payment_type'] = taxi_data['payment_type'].astype('int64')

In [ ]:
# check for the duplicate rows 
taxi_data[taxi_data.duplicated()]

In [ ]:
# removing duplicate rows as they will not contribute in analysis 
taxi_data.drop_duplicates(inplace = True)

In [ ]:
# after removing missing values and duplicate rows, remaining data:
taxi_data.shape


In [ ]:
# passenger count distribution
taxi_data['passenger_count'].value_counts(normalize = True)

In [ ]:
# payment type distribution
taxi_data['payment_type'].value_counts(normalize = True)

### We will focus solely on payment types 'card' and 'cash,' denoted by 1 and 2 in the dataset. To ensure our analysis centers on these payment methods, we'll filter the data accordingly, excluding all other types. so we need to keep only 1 and 2 payment type.

### As well as, examining the distribution of passenger counts shows that trips with more than 5 passengers are rare. Additionally, trips with 0 passengers are impossible, as we expect at least one passenger to pay the fare amount. Therefore, we will filter the passenger count to include only values ranging from 1 to 5.

In [ ]:
# filtering for payment type 1 and 2
taxi_data =taxi_data[taxi_data['payment_type']<3]

# filtering for passenger count from 1 to 5
taxi_data = taxi_data[(taxi_data['passenger_count']>0) & (taxi_data['passenger_count']<6)]

In [ ]:
# remaining data
taxi_data.shape

In [ ]:
# replacing the payment type encoded value 1 and 2 to Card and Cash
taxi_data['payment_type'].replace([1,2],['Card','Cash'], inplace = True)

In [ ]:
# descriptive statistics for data
taxi_data.describe()

In [ ]:
# there is a problem: 'fare_amount' column dosen't appear in descriptive data. we need to change data type of this column

# check data type of 'fare_amount' column
taxi_data.dtypes  #output = object

#convert it into numeric
taxi_data['fare_amount'] = pd.to_numeric(taxi_data['fare_amount'],errors='coerce') # why 'errors = coerce'?  it converts invalid values such as 'NA' into 'NaN', which you can then handle.

#Then check: 
taxi_data['fare_amount'].dtypes

In [ ]:
# descriptive statistics for data
taxi_data.describe()

### Upon reviewing the provided statistics, it's evident that the minimum values for trip distance, fare amount, and duration are negative, which is unrealistic and invalid for further analysis. Therefore, we will eliminate these negative values from the dataset.

### Also, observing the maximum and 75th percentile values, it's possible that the data contains significant outliers, particularly high values. These outliers need to be addressed and removed to ensure the integrity of the analysis.

In [ ]:
# filtering the records for only positive values
taxi_data = taxi_data[taxi_data['fare_amount']> 0]
taxi_data = taxi_data[taxi_data['trip_distance']>0]
taxi_data = taxi_data[taxi_data['duration']>0]

In [ ]:
 # check for the outliers
sns.boxplot(data = taxi_data, y = "fare_amount", x = "payment_type")
plt.show()

In [ ]:
plt.boxplot(taxi_data['fare_amount'])

In [ ]:
# removing outliers using interquartile range for the numerical variables
for col in ['trip_distance','fare_amount','duration']:
    Q1 = taxi_data[col].quantile(0.25)
    Q3 = taxi_data[col].quantile(0.75)
    IQR = Q3 - Q1
    
# Define lower and upper bounds (whisker) for outliers
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
# Filter out outliers
    taxi_data = taxi_data[(taxi_data[col] >= lower_bound) & (taxi_data[col] <= upper_bound)]

In [ ]:
taxi_data.shape

### We're interested on exploring the relationship between payment type and passenger behavior concerning trip distance and fare amount. Are there variations in the distribution of payment types concerning different fare amounts or trip distances?

### To investigate this, we'll plot histograms to visualize the distribution of passenger counts paying with either card or cash. This will also provide stakeholders with insight into fare amount ranges associated with different payment methods.

In [ ]:
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.title('Distribution of Fare Amount')
plt.hist(taxi_data[taxi_data['payment_type']=='Card']['fare_amount'], histtype='barstacked', bins = 20, edgecolor = 'k', color = '#FA643F', label = 'Card')
plt.hist(taxi_data[taxi_data['payment_type']=='Cash']['fare_amount'],  histtype='barstacked',bins = 20, edgecolor = 'k', color = '#FFBCAB',label = 'Cash')
plt.legend()

plt.subplot(1,2,2)
plt.title('Distribution of Trip Distance')
plt.hist(taxi_data[taxi_data['payment_type']=='Card']['trip_distance'], histtype='barstacked', bins = 20, edgecolor = 'k',color = '#FA643F',label = 'Card')
plt.hist(taxi_data[taxi_data['payment_type']=='Cash']['trip_distance'],  histtype='barstacked',bins = 20, edgecolor = 'k', color = '#FFBCAB',label = 'Cash')
plt.legend()
plt.show()

# calculating the mean and standard deviation group by on payment type 
taxi_data.groupby('payment_type').agg({'fare_amount': ['mean', 'std',], 'trip_distance': ['mean', 'std'],})

### Now, in order to examine the passenger's preference regarding their choice of payment method, we will assess the proportion of the two payment types. To provide a visual representation, we have opted to utilize a pie chart. This graphical depiction will offer a clear and intuitive understanding of the distribution between the two payment methods chosen by passengers.


In [ ]:
plt.title('Preference of Payment Type')
plt.pie(taxi_data['payment_type'].value_counts(normalize = True), labels = taxi_data['payment_type'].value_counts().index,
       startangle = 90,shadow = True, autopct = '%1.1f%%', colors = ['#FA643F','#FFBCAB'])
plt.show()

### Subsequently, we aim to conduct an analysis of the payment types in relation to the passenger count. Our objective is to investigate if there are any changes in preference contingent upon the number of passengers traveling in the cab. 

### To facilitate this examination, we have employed a visualization technique known as a stacked bar plot. This method is particularly advantageous for comparing the percentage distribution of each passenger count based on the payment method selected. Through this graphical representation, we can gain insights into potential variations in payment preferences across different passenger counts.

In [ ]:
# calculating the total passenger count distribution based on the different payment type
passenger_count = taxi_data.groupby(['payment_type','passenger_count'])[['passenger_count']].count()

# renaming the passenger_count to count to reset the index
passenger_count.rename(columns = {'passenger_count':'count'},inplace = True)
passenger_count.reset_index(inplace = True)

# calculating the percentage of the each passenger count
passenger_count['perc'] = (passenger_count['count']/passenger_count['count'].sum())*100

# creating a new empty dataframe to store the distribution of each payment type (useful for the visualization)
df = pd.DataFrame(columns = ['payment_type',1,2,3,4,5])
df['payment_type'] = ['Card','Cash']
df.iloc[0,1:] = passenger_count.iloc[:5,-1]
df.iloc[1,1:] = passenger_count.iloc[5:,-1]

fig, ax = plt.subplots(figsize=(20, 6)) 
df.plot(x='payment_type', kind='barh', stacked=True, title=' ', ax=ax, color = ['#FA643F', '#FFBCAB', '#CBB2B2', '#F1F1F1', '#FD9F9F'] )

# Add percentage text
for p in ax.patches:
    width = p.get_width()
    height = p.get_height()
    x, y = p.get_xy()
    ax.text(x + width / 2,
            y + height / 2,
            '{:.0f}%'.format(width),
            horizontalalignment='center',
            verticalalignment='center')

## REGRESSION ANALYSIS
## Relationship between Trip Duration and Fare Amount

In [ ]:
# Select variables for regression
reg_data = taxi_data[['duration', 'fare_amount']].dropna()

# Independent variable (X)
X = reg_data['duration']

# Dependent variable (Y)
y = reg_data['fare_amount']

# Add constant/intercept
X = sm.add_constant(X)

# Fit the linear regression model
model = sm.OLS(y, X).fit()

# Display regression results
print(model.summary())

In [ ]:
# Scatter plot with regression line

plt.figure(figsize=(10, 6))

sns.regplot(
    data=reg_data,
    x='duration',
    y='fare_amount',
    scatter_kws={'alpha': 0.2},
    line_kws={'color': 'red'}
)

plt.title('Relationship Between Trip Duration and Fare Amount')
plt.xlabel('Trip Duration (Minutes)')
plt.ylabel('Fare Amount ($)')
plt.show()

In [ ]:
# Regression coefficients and statistics

print("Intercept:", model.params['const'])
print("Duration Coefficient:", model.params['duration'])
print("R-squared:", model.rsquared)
print("Adjusted R-squared:", model.rsquared_adj)
print("P-value:", model.pvalues['duration'])

In [ ]:
regression_summary = pd.DataFrame({
    'Metric': [
        'Intercept',
        'Duration Coefficient',
        'R-squared',
        'Adjusted R-squared',
        'P-value'
    ],
    'Value': [
        model.params['const'],
        model.params['duration'],
        model.rsquared,
        model.rsquared_adj,
        model.pvalues['duration']
    ]
})

regression_summary

### Interpretation:
### - The regression analysis indicates a statistically significant positive relationship between trip duration and fare amount. The positive coefficient suggests that, on average, fare amount increases as trip duration increases. Since the p-value is below 0.05, trip duration is a statistically significant predictor of fare amount.

## Hypothesis Testing

### In order to select the most suitable test for our scenario, our initial step involves evaluating whether the distribution of fare amounts adheres to a normal distribution. While the histogram depicted above suggests otherwise, we will further confirm this by generating a QQ plot.

### Quantile-quantile (QQ) plots can be used to assess whether the fare amount distributions for each payment type are approximately normally distributed. If the data points closely align with the diagonal line in the plot, it suggests that the data follows a normal distribution.

In [ ]:
#create Q-Q plot with 45-degree line added to plot
fig = sm.qqplot(taxi_data['fare_amount'], line='45')
plt.show()

### The data values clearly do not follow the red 45-degree line, which is an indication that they do not follow a normal distribution. So, z distribution will not be good for this. That's why we will use T test.

### Given that the T-test can be applied to both small and large samples and does not require the population standard deviation, it is a more universally applicable approach for hypothesis testing in many practical research scenarios, including analyses of taxi trip data.


### In the analysis of Taxi Trip Records, where you're likely dealing with an unknown population standard deviation and potentially large datasets, the T-test offers a more appropriate and flexible method for comparing means between two groups (e.g., fare amounts by payment type). It provides a reliable way to infer about the population, accommodating the uncertainty that comes with estimating population parameters from sample data.

**Null hypothesis**: There is no difference in average fare between customers who use credit cards and customers who use cash. 

**Alternative hypothesis**: There is a difference in average fare between customers who use credit cards and customers who use cash



In [ ]:
# sample 1
credit_card = taxi_data[taxi_data['payment_type'] == 'Card']['fare_amount']

# sample 2
cash = taxi_data[taxi_data['payment_type'] == 'Cash']['fare_amount']

# performing t test on both the different sample
t_stat, p_value = stats.ttest_ind(a=credit_card, b=cash, equal_var=False)
print(f"T-statistic: {t_stat}, P-value: {p_value:.5f}")

# comparing the p value with the significance of 5% or 0.05
if p_value < 0.05:
    print("\nReject the null hypothesis")
else:
    print("\nAccept the null hypothesis")

### Since the p-value is significantly smaller than the significance level of 5%, we will reject the null hypothesis.

### You conclude that there is a statistically significant difference in the average fare amount between customers who use credit cards and customers who use cash.

### The key business insight is that encouraging customers to pay with credit cards can generate more revenue for taxi cab drivers.

## Recommendations

- Encourage customers to pay with credit cards to capitalize on the potential for generating more revenue for taxi cab drivers
- Implement strategies such as offering incentives or discounts for credit card transactions to incentivize customers to choose this payment method.
- Provide seamless and secure credit card payment options to enhance customer convenience and encourage adoption of this preferred payment method